# example / wasm / accumulate

Add a fitted line to each input and keep a running total and peak, as a WebAssembly module that also exercises the tracer and every host maths function.

| Term | In This Notebook |
|---|---|
| **Intent** | Show a stateful `wasm` model end to end: fit, trace the module, accept, replay the golden and register. It doubles as the test of the tracer and the three WebAssembly hosts: the module uses every operator and helper of `workbench.wasm` and imports all fifteen host maths functions, and the golden is replayed in Python, Go and Rust |
| **Project Flavour** | `wasm` (`model.md` 3.2) |
| **Data Splits** | `training` ten labelled points on a line with fixed noise, to fit; `validation` five more; `backtest` ten further out, held back for acceptance; `golden` the exact samples that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, version and dataset names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

import numpy as np

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "example/wasm/accumulate"
MODEL_NAME = "example_wasm_accumulate"
TARGETS = [{"kind": "served"}, {"kind": "embedded", "language": "python"}, {"kind": "embedded", "language": "go"}, {"kind": "embedded", "language": "rust"}]
TRAINING = "wasm_accumulate_training_2026-11-03_2026-11-03"
VALIDATION = "wasm_accumulate_validation_2026-11-03_2026-11-03"
BACKTEST = "wasm_accumulate_backtest_2026-11-03_2026-11-03"
GOLDEN = "wasm_accumulate_golden"
INPUT = "example_wasm_in"
OUTPUT = "example_wasm_out"

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, so no query runs.

In [ ]:
NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [float(x) for x in range(first, first + count)]
    return {INPUT: xs, OUTPUT: [round(2.5 * x + 1.0 + NOISE[int(x) % 10], 4) for x in xs]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled(0, 10))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled(10, 5))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled(15, 10))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** fit a straight line on `training` only (R6). Its gain and offset are rounded to one place, which the module builds in.

In [ ]:
gain, offset = np.polyfit(training_pl[INPUT].to_numpy(), training_pl[OUTPUT].to_numpy(), 1)
GAIN, OFFSET = round(float(gain), 1), round(float(offset), 1)
print("fitted gain", GAIN, "offset", OFFSET)
assert (GAIN, OFFSET) == (2.5, 1.0), "the fit did not recover the line that made the data"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** trace `accumulate`, the fitted line, a running total and peak and a set of checks written once as a Python function, into the module (`model.md` 4.4), build the signature, run the module through the library host and check the `validation` and `backtest` error of the line against the limit the notebook owns, and of every check output against the plain Python call of the same function, which gates the tracer (R4, R14).

In [ ]:
from workbench import wasm
from workbench.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output, signature_state
from workbench.replay import model_predict

LOW, HIGH = 5.0, 20.0
UNARY = ("acos", "asin", "atan", "cos", "cosh", "exp", "log", "log10", "log2", "sin", "sinh", "tan", "tanh")
STATE_IN = "example_wasm_state_in"
STATE_OUT = "example_wasm_state_out"


def exercise(x, out):
    unit = wasm.clip(x / 16.0, 0.0, 1.0)
    checks = {
        "bounded": wasm.clip(out, LOW, HIGH),
        "band": wasm.select(out < LOW, -1.0, wasm.select(out > HIGH, 1.0, 0.0)),
        "band_inclusive": wasm.select(out <= LOW, 0.0, wasm.select(out >= HIGH, 2.0, 1.0)),
        "equal": wasm.select(wasm.eq(out, 8.5), 1.0, 0.0),
        "unequal": wasm.select(wasm.ne(out, 8.5), 1.0, 0.0),
        "floor": wasm.floor(out),
        "ceil": wasm.ceil(out),
        "trunc": wasm.trunc(-out),
        "nearest": wasm.nearest(out),
        "magnitude": wasm.sqrt(abs(-out)),
        "polynomial": 0.01 * out**3 - 0.2 * out**2 + out,
        "power_const": out**1.5,
        "power_base": 2.0**unit,
        "power_unit": wasm.power(unit, 2.5),
        "atan2": wasm.atan2(out, x),
    }
    return checks | {name: getattr(wasm, name)(unit) for name in UNARY}


def accumulate(x, total, peak, gain, offset):
    out = gain * x + offset
    return [out, *exercise(x, out).values(), total + out, wasm.maximum(peak, out)]


CHECKS = list(exercise(1.0, 3.5))
TOLERANT = [name for name in CHECKS if name in (*UNARY, "polynomial", "power_const", "power_base", "power_unit", "atan2")]

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.wasm")
with open(model_path, "wb") as model_file:
    model_file.write(wasm.build_wasm(3, wasm.trace(lambda x, total, peak: accumulate(x, total, peak, GAIN, OFFSET), 3)))

signature = build_signature(
    MODEL_NAME,
    [
        signature_input(INPUT, "none", "A number to scale."),
        signature_input(STATE_IN, "none", "The running total and peak from the previous call, absent on the first.", required=False, shape=[-1]),
    ],
    [
        signature_output(OUTPUT, "none", "The input scaled by the fitted gain plus the fitted offset.", datatype="FP64"),
        *[signature_output(f"example_wasm_{name}", "none", f"The {name} check of the tracer and the hosts.", datatype="FP64") for name in CHECKS],
        signature_output(STATE_OUT, "none", "The running total and peak including this call.", datatype="FP64", required=False, shape=[-1]),
    ],
    state=[
        signature_state("total", "none", "The sum of every output since the first call."),
        signature_state("peak", "none", "The largest output since the first call."),
    ],
)
predict = model_predict(model_path, "wasm", signature)

In [ ]:
def acceptance_checks():
    checks = []
    for split, frame in (("validation", validation_pl), ("backtest", backtest_pl)):
        xs = frame[INPUT].to_list()
        predicted = [predict({INPUT: [x]}) for x in xs]
        checks.append(
            {"split": split, "output": OUTPUT, "name": "rmse", "value": measure(frame[OUTPUT].to_list(), [answer[OUTPUT][0] for answer in predicted])["rmse"], "limit": 0.5, "direction": "max"}
        )
        for name in CHECKS:
            plain = [exercise(x, GAIN * x + OFFSET)[name] for x in xs]
            checks.append(
                {
                    "split": split,
                    "output": f"example_wasm_{name}",
                    "name": "rmse",
                    "value": measure(plain, [answer[f"example_wasm_{name}"][0] for answer in predicted])["rmse"],
                    "limit": 1e-9,
                    "direction": "max",
                }
            )
    return checks


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "limit", entry["limit"], "passed", entry["passed"])

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file, copy the module and the evaluation into a project, verify it and replay the golden through the library host (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [1.0, 3.0, 8.0, 10.0, 14.0, 2.0]
    names = [OUTPUT, *[f"example_wasm_{name}" for name in CHECKS]]
    rows, total, peak = [], 0.0, 0.0
    for index, x in enumerate(xs):
        *scalars, next_total, next_peak = accumulate(x, total, peak, GAIN, OFFSET)
        rows.append({INPUT: x, STATE_IN: None if index == 0 else f"{total!r} {peak!r}", **dict(zip(names, scalars, strict=True)), STATE_OUT: f"{next_total!r} {next_peak!r}", "expect": "ok"})
        total, peak = next_total, next_peak
    rows.append({"expect": "error"})
    return {key: [row.get(key) for row in rows] for key in (INPUT, STATE_IN, *names, STATE_OUT, "expect")}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "wasm",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic line, training split",
        [training_path, golden_path],
        "least squares line, gain and offset built into the module",
        implementation="WebAssembly traced from a Python function by workbench.wasm.trace and built by build_wasm, importing every asystem_math function",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="a running total and peak over six calls",
    tolerance={f"example_wasm_{name}": {"absolute": 1e-9, "relative": 1e-9} for name in TOLERANT},
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)